In [ ]:
"""
Understand from the initial weight transfer where the discrepency is in the weight transfer between TF and PT 
"""
import torch
import random 
import numpy as np
import predictor_tf
import seisbench.data as sbd
from predictor_pt import EQCCTModelS

np.random.seed(42)

def trace_s_model_path(flip=None):
    """
    Performs a full forward pass, printing tensor statistics at every
    major step to find the exact point of failure.
    """
    print("=== TRACING TENSOR PATH THROUGH S-MODEL ===\n")

    # --- Load Model and Input ---
    # modelS_pt = EQCCTModelS()
    # modelS_pt.load_state_dict(torch.load('new_smodel.pth'))
    modelS = flip
    modelS_pt.eval()
    
    # Get input from TXED 
    txed_datasource = sbd.TXED
    txed = txed_datasource(sampling_rate=100, component_order='ZNE', cache=None, metadata_cache=False)
    md = txed.metadata
    
    # Filter for waveforms that have P and S picks 
    good_p = md["trace_p_arrival_sample"].notna() & (md["trace_p_arrival_sample"] > 0)
    good_s = md["trace_s_arrival_sample"].notna() & (md["trace_s_arrival_sample"] > 0)
    valid = md[good_p & good_s]["trace_name"].tolist()

    # Random shuffle 
    random.shuffle(valid)
    example = valid[9]
    example_metadata = txed.metadata[txed.metadata['trace_name'].str.contains(example, na=False, regex=False)]
    index = example_metadata.index[0]
    traces, labels = txed.get_sample(index)

    # test_input = torch.from_numpy(traces).unsqueeze(0).float() # 
    test_input = torch.from_numpy(np.random.randn(1, 6000, 3).astype('float32') * 0.5)

    # --- Helper Function to Print Stats ---
    def print_stats(tensor, name):
        mean = tensor.mean().item()
        std = tensor.std().item()
        abs_max = tensor.abs().max().item()
        print(f"{name:<28} | Mean: {mean:>8.4f} | Std: {std:>8.4f} | Max Abs: {abs_max:>8.4f}")

    # --- Manually Trace the Forward Pass ---
    with torch.no_grad():
        x = test_input
        print_stats(x, "Initial Input")
        print("-" * 70)

        x = x.transpose(1, 2) # Switch to (B, C, L)

        x = modelS_pt.conv1(x)
        print_stats(x, "After conv1 block")
        x = modelS_pt.conv2(x)
        print_stats(x, "After conv2 block")
        x = modelS_pt.conv3(x)
        print_stats(x, "After conv3 block")
        print("-" * 70)

        x = x.unsqueeze(2).permute(0, 3, 2, 1) # Reshape for patching
        x = modelS_pt.patch(x)
        print_stats(x, "After Patches")
        x = modelS_pt.encoder(x)
        print_stats(x, "After PatchEncoder")
        print("=" * 70)

        # Trace through each Transformer Block
        for i in range(4):
            print(f"TRANSFORMER BLOCK {i}")
            
            # Pre-Conv
            identity_skip2 = x
            x = x.transpose(1, 2)
            x = getattr(modelS_pt, f'extra_pre')[i](x)
            x = x.transpose(1, 2)
            print_stats(x, f"  -> extra_pre.{i}")
            
            identity_skip1 = x
            
            # Attention Path
            x = getattr(modelS_pt, 'transformers')[i].norm1(x)
            print_stats(x, f"  -> norm1.{i}")
            x = getattr(modelS_pt, 'transformers')[i].attn(x)
            print_stats(x, f"  -> attention.{i}")

            # Post-Conv
            x = x.transpose(1, 2)
            x = getattr(modelS_pt, f'extra_post')[i](x)
            x = x.transpose(1, 2)
            print_stats(x, f"  -> extra_post.{i}")
            
            # Skip Connections
            x = identity_skip1 + getattr(modelS_pt, 'transformers')[i].drop_path1(x)
            print_stats(x, f"  -> After Skip 1.{i}")

            x_mlp = getattr(modelS_pt, 'transformers')[i].norm2(x)
            x_mlp = getattr(modelS_pt, 'transformers')[i].mlp(x_mlp)
            x = x + getattr(modelS_pt, 'transformers')[i].drop_path2(x_mlp)
            print_stats(x, f"  -> After Skip 2.{i}")
            print("-" * 70)

        # Final Layers
        x = modelS_pt.norm(x)
        print_stats(x, "After final LayerNorm")
        
        x = x.reshape(x.size(0), 6000, 1)
        x = x.transpose(1, 2)
        x = modelS_pt.head.conv(x)
        x = x.transpose(1, 2)
        print_stats(x, "Final Output (pre-sigmoid)")
        print("=" * 70)


# if __name__ == "__main__":
#     trace_s_model_path()

=== TRACING TENSOR PATH THROUGH S-MODEL ===



RuntimeError: Error(s) in loading state_dict for EQCCTModelS:
	Missing key(s) in state_dict: "extra_pre.0.conv1.weight", "extra_pre.0.conv1.bias", "extra_pre.0.bn1.weight", "extra_pre.0.bn1.bias", "extra_pre.0.bn1.running_mean", "extra_pre.0.bn1.running_var", "extra_pre.0.conv2.weight", "extra_pre.0.conv2.bias", "extra_pre.0.bn2.weight", "extra_pre.0.bn2.bias", "extra_pre.0.bn2.running_mean", "extra_pre.0.bn2.running_var", "extra_pre.0.conv3.weight", "extra_pre.0.conv3.bias", "extra_pre.0.bn3.weight", "extra_pre.0.bn3.bias", "extra_pre.0.bn3.running_mean", "extra_pre.0.bn3.running_var", "extra_pre.1.conv1.weight", "extra_pre.1.conv1.bias", "extra_pre.1.bn1.weight", "extra_pre.1.bn1.bias", "extra_pre.1.bn1.running_mean", "extra_pre.1.bn1.running_var", "extra_pre.1.conv2.weight", "extra_pre.1.conv2.bias", "extra_pre.1.bn2.weight", "extra_pre.1.bn2.bias", "extra_pre.1.bn2.running_mean", "extra_pre.1.bn2.running_var", "extra_pre.1.conv3.weight", "extra_pre.1.conv3.bias", "extra_pre.1.bn3.w

: 